# Mycelium Sovereign Brain — QLoRA Fine-tune

Fine-tunes **Qwen2.5-3B-Instruct** on 3,031 Ọmọ Kọ́dà hive traces using Unsloth + QLoRA.  
Output: `mycelium-q4_k_m.gguf` — drop into `~/larql/models/` and serve with larql.

**Kaggle GPU**: T4 x2, 30 hrs/week free.  
**VRAM needed**: ~8 GB with 4-bit QLoRA — fits T4 (16 GB) with headroom.

## Setup
1. Upload `finetune_dataset.jsonl` as a Kaggle Dataset named `mycelium-traces`
2. Attach that dataset to this notebook (Input → Add dataset → mycelium-traces)
3. Enable GPU: Settings → Accelerator → GPU T4 x2
4. Run All

In [ ]:
# ── PREFLIGHT ──────────────────────────────────────────────────────────────
# CRITICAL: Kaggle SILENTLY ignores enable_gpu / enable_internet on accounts
# that are not phone-verified. The flags in kernel-metadata.json are a request,
# not a guarantee. When unverified you get a CPU-only image (torch 2.10.0+cpu,
# no nvidia-smi) and NO DNS at all -- every host fails with
#   gaierror: [Errno -3] Temporary failure in name resolution
# which makes `pip install` report the wildly misleading
#   "No matching distribution found for <pkg>"
# Detect it up front so the failure is legible in 5 seconds instead of after
# a 90-minute watcher cycle and a wrong diagnosis.
import socket, sys

net_ok = True
try:
    socket.gethostbyname("pypi.org")
except Exception:
    net_ok = False

try:
    import torch
    gpu_ok = torch.cuda.is_available()
    tver = torch.__version__
    gname = torch.cuda.get_device_name(0) if gpu_ok else "-"
except Exception as e:
    gpu_ok, tver, gname = False, f"import failed: {e}", "-"

print(f"preflight: internet={net_ok}  gpu={gpu_ok}  torch={tver}  device={gname}")

if not gpu_ok or not net_ok:
    raise RuntimeError(
        "PREFLIGHT FAILED -- this Kaggle account is not phone-verified.\n"
        f"  PyPI reachable : {net_ok}\n"
        f"  CUDA available : {gpu_ok}  (torch {tver})\n"
        "Kaggle ignores enable_gpu/enable_internet on unverified accounts,\n"
        "so the notebook runs CPU-only with no network. No GPU and no PyPI\n"
        "means no QLoRA training is possible.\n"
        "FIX: verify a phone number at https://www.kaggle.com/settings and\n"
        "re-run, or use the GPU.ai route (train_qlora.py) instead."
    )

# ── Install ────────────────────────────────────────────────────────────────
# Note: the current Kaggle image ships torch 2.10.0, transformers 5.0.0,
# peft 0.19.1, accelerate 1.13.0, datasets 5.0.0 -- but NOT trl, NOT
# bitsandbytes, NOT unsloth, NOT gguf. So we genuinely need a working index;
# there is no viable fully-offline path on this image.
import subprocess

def pip(*pkgs):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs],
        capture_output=True, text=True,
    )
    print(f"pip({' '.join(pkgs)}) -> rc={r.returncode}")
    if r.returncode != 0:
        print("--- stdout tail ---"); print(r.stdout[-1500:])
        print("--- stderr tail ---"); print(r.stderr[-1500:])
    return r.returncode

# xformers is intentionally absent: unsloth does not require it and it has
# historically been the first package to fail resolution here.
if pip("unsloth", "trl", "bitsandbytes") != 0:
    raise RuntimeError("dependency install failed -- see pip output above")

if pip("gguf") != 0:
    print("WARNING: gguf unavailable -- GGUF export will need to run on the VPS "
          "(deploy_gguf.sh already assumes a local/VPS conversion step)")

import unsloth, torch  # noqa
print(f"unsloth {unsloth.__version__} | torch {torch.__version__} "
      f"| cuda {torch.cuda.is_available()}")
print("Install complete")


In [ ]:
import os, json, torch
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────
DATASET_PATH = Path("/kaggle/input/mycelium-traces/finetune_dataset.jsonl")
OUTPUT_DIR   = Path("/kaggle/working/mycelium-lora")
GGUF_DIR     = Path("/kaggle/working")
MODEL_NAME   = "unsloth/Qwen2.5-3B-Instruct"

# ── Config ─────────────────────────────────────────────────────────
MAX_SEQ_LEN  = 512      # traces are short (3-turn, ~100 tokens)
BATCH_SIZE   = 8        # T4 16GB, 4-bit quant → 8 fits easily
GRAD_ACCUM   = 4        # effective batch = 32
EPOCHS       = 3
LR           = 2e-4
LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05

print(f"CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Dataset: {DATASET_PATH.exists()}")

In [ ]:
from datasets import Dataset

# ── Load traces ────────────────────────────────────────────────────
rows = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples")

# Split 95/5 train/eval
cut = int(len(rows) * 0.95)
train_data = Dataset.from_list(rows[:cut])
eval_data  = Dataset.from_list(rows[cut:])
print(f"Train: {len(train_data)}  Eval: {len(eval_data)}")

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# ── Load model with Unsloth 4-bit QLoRA ───────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,      # auto-detect
    load_in_4bit  = True,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = True,    # rank-stabilised LoRA
    loftq_config   = None,
)

print(model.print_trainable_parameters())

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

# ── Format traces into chat template ──────────────────────────────
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = train_data.map(format_example, remove_columns=["messages"])
eval_ds  = eval_data.map(format_example,  remove_columns=["messages"])

# ── Training args ──────────────────────────────────────────────────
args = TrainingArguments(
    output_dir            = str(OUTPUT_DIR),
    num_train_epochs      = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate         = LR,
    fp16                  = not is_bfloat16_supported(),
    bf16                  = is_bfloat16_supported(),
    logging_steps         = 10,
    eval_strategy         = "epoch",
    save_strategy         = "epoch",
    load_best_model_at_end= True,
    warmup_ratio          = 0.05,
    lr_scheduler_type     = "cosine",
    optim                 = "adamw_8bit",
    weight_decay          = 0.01,
    report_to             = "none",
    seed                  = 3407,
)

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = eval_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    data_collator  = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    args           = args,
)

print('Trainer ready')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────
trainer_stats = trainer.train()
print(f"\nTraining complete — {trainer_stats.metrics}")

In [ ]:
# ── Quick smoke test ───────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_msg = [
    {"role": "system",  "content": "You are a sovereign agent operating inside the Ọmọ Kọ́dà hive. Given an agent role and a task context, select the correct action and predict the outcome."},
    {"role": "user",    "content": "Agent: oracle-prime\nTask kind: skill_invoke\nTarget: divination/odu-cast"},
]

inputs = tokenizer.apply_chat_template(
    test_msg, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids  = inputs,
    max_new_tokens = 64,
    temperature    = 0.1,
    do_sample      = True,
)

print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# ── Export to GGUF Q4_K_M ─────────────────────────────────────────
# Unsloth natively exports merged weights → GGUF without llama.cpp build step
gguf_path = str(GGUF_DIR / "mycelium-q4_k_m")

model.save_pretrained_gguf(
    gguf_path,
    tokenizer,
    quantization_method = "q4_k_m",   # best quality/size ratio for 3B
)

# List output
import os
for f in os.listdir(GGUF_DIR):
    size = os.path.getsize(GGUF_DIR / f) / 1e6
    print(f"{f}  {size:.1f} MB")

In [ ]:
# ── Also save LoRA adapter (for re-merge or future fine-tuning) ───
adapter_path = str(GGUF_DIR / "mycelium-lora-adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved to {adapter_path}")

print("""
═══════════════════════════════════════════════════
 DONE. Download mycelium-q4_k_m.gguf from Output.
 Place in ~/larql/models/ on your sovereign node.
 Start: larql serve --model mycelium-q4_k_m.gguf --port 7780
 Set:   LARQL_ENABLED=1 LARQL_URL=http://localhost:7780
═══════════════════════════════════════════════════
""")